# 🏥 Medical Vision AI Agent — Training Notebook

## Overview
This notebook trains a **complete Medical Vision AI Agent** for skin disease classification.

### Architecture
- **EfficientNet-B3** — State-of-the-art efficient convolutional network
- **MobileNetV2** — Lightweight mobile architecture
- **ResNet50** — Classic deep residual network
- **Ensemble Logic**: All agree → average confidence | Disagree → highest confidence wins
- **VLM Explanation**: Natural language explanation of each diagnosis

### Dataset
Uses the [Skin Disease Dataset](https://www.kaggle.com/datasets/ismailpromus/skin-diseases-image-dataset) with 10 classes.

### Medical Disclaimer
> ⚠️ **This is not medical advice. Consult a healthcare professional.**
> This system is for research and educational purposes only.

## 1. Setup — Install Dependencies & Clone Repo

In [1]:
# Install required packages
!pip install -q torch torchvision Pillow matplotlib seaborn tqdm transformers

# Check GPU availability
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

🖥️  Device: cuda
   GPU: NVIDIA A100-SXM4-40GB
   Memory: 42.4 GB


In [2]:
# Mount Google Drive (to save models persistently)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/medical_vision_ai'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Drive mounted. Models will be saved to: {DRIVE_DIR}')

Mounted at /content/drive
✅ Drive mounted. Models will be saved to: /content/drive/MyDrive/medical_vision_ai


In [3]:
# Clone the repository
!git clone https://github.com/GoatAbdouu/medical_agent_starter.git /content/medical_agent_starter
%cd /content/medical_agent_starter

import sys
sys.path.insert(0, '/content/medical_agent_starter')
print('✅ Repository cloned and added to Python path')

Cloning into '/content/medical_agent_starter'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 124 (delta 49), reused 88 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.57 MiB | 6.07 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/medical_agent_starter
✅ Repository cloned and added to Python path


## 2. Dataset Setup — Skin Disease Images

In [4]:
# Option A: Download from Kaggle
# First upload your kaggle.json credentials file
from google.colab import files
# Uncomment to upload kaggle.json:
# uploaded = files.upload()  # Upload kaggle.json

!mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d ismailpromus/skin-diseases-image-dataset
# !unzip -q skin-diseases-image-dataset.zip -d /content/skin_data

# Option B: Mount from Drive (if already downloaded)
# DATA_DIR = '/content/drive/MyDrive/skin_dataset/IMG_CLASSES'

# Set DATA_DIR to your dataset path:
DATA_DIR = '/content/drive/MyDrive/medical_agent/dataset/IMG_CLASSES'  # Adjust if needed
print(f'DATA_DIR = {DATA_DIR}')

DATA_DIR = /content/skin_data/IMG_CLASSES


In [5]:
# Explore dataset distribution
import os
import matplotlib.pyplot as plt
from medical_agent.core.skin_disease_classifier import FOLDER_TO_DISEASE

if os.path.exists(DATA_DIR):
    class_folders = sorted(os.listdir(DATA_DIR))
    class_counts = {}
    for folder in class_folders:
        folder_path = os.path.join(DATA_DIR, folder)
        if os.path.isdir(folder_path):
            count = len([f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            readable = FOLDER_TO_DISEASE.get(folder, folder)
            class_counts[readable] = count

    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.barh(list(class_counts.keys()), list(class_counts.values()), color='steelblue')
    ax.set_xlabel('Number of Images')
    ax.set_title('Skin Disease Dataset Distribution')
    for bar, count in zip(bars, class_counts.values()):
        ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
                str(count), va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f'\nTotal images: {sum(class_counts.values()):,}')
else:
    print(f'⚠️ Dataset not found at {DATA_DIR}. Please set DATA_DIR correctly.')

⚠️ Dataset not found at /content/skin_data/IMG_CLASSES. Please set DATA_DIR correctly.


## 3. Image Pipeline — Augmentation Demo

In [6]:
import os, random
import matplotlib.pyplot as plt
from PIL import Image
from medical_agent.core.image_pipeline import ImagePipeline

pipeline = ImagePipeline()
train_tf = pipeline.get_train_transforms()
val_tf = pipeline.get_val_transforms()

print('Train transforms:', train_tf)
print('\nVal transforms:', val_tf)

# Show augmentation examples if dataset is available
if os.path.exists(DATA_DIR):
    # Pick a random image
    folders = [f for f in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, f))]
    folder = random.choice(folders)
    images = [f for f in os.listdir(os.path.join(DATA_DIR, folder))
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    img_path = os.path.join(DATA_DIR, folder, random.choice(images))
    orig_img = Image.open(img_path).convert('RGB')

    import torchvision.transforms.functional as F
    import numpy as np

    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    fig.suptitle(f'Augmentation Examples — {FOLDER_TO_DISEASE.get(folder, folder)}', fontsize=14)
    axes[0, 0].imshow(orig_img.resize((224, 224)))
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')

    # Show 9 augmented versions
    from torchvision import transforms
    to_pil = transforms.ToPILImage()
    unnorm = transforms.Compose([
        transforms.Normalize(mean=[0, 0, 0], std=[1/0.229, 1/0.224, 1/0.225]),
        transforms.Normalize(mean=[-0.485, -0.456, -0.406], std=[1, 1, 1]),
    ])
    flat_axes = [axes[r][c] for r in range(2) for c in range(5)][1:]
    for ax in flat_axes:
        aug = train_tf(orig_img)
        img_vis = to_pil(unnorm(aug).clamp(0, 1))
        ax.imshow(img_vis)
        ax.set_title('Augmented')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

Train transforms: Compose(
    Resize(size=(256, 256), interpolation=bilinear, max_size=None, antialias=True)
    RandomResizedCrop(size=(224, 224), scale=(0.7, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    RandomRotation(degrees=[-30.0, 30.0], interpolation=nearest, expand=False, fill=0)
    RandomAffine(degrees=[0.0, 0.0], translate=(0.1, 0.1), shear=[-10.0, 10.0])
    ColorJitter(brightness=(0.7, 1.3), contrast=(0.7, 1.3), saturation=(0.7, 1.3), hue=(-0.1, 0.1))
    GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0))
    ToTensor()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
)

Val transforms: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
)


## 4. Model Architecture Overview

All three models share a common deep classifier head:
```
in_features → Linear(512) → BatchNorm → ReLU → Dropout(0.4)
           → Linear(256) → BatchNorm → ReLU → Dropout(0.3)
           → Linear(num_classes)
```

| Model | Backbone | Head Input Features | Parameters |
|-------|----------|--------------------|-----------|
| EfficientNet-B3 | `efficientnet_b3` | 1536 | ~12M |
| MobileNetV2 | `mobilenet_v2` | 1280 | ~3.4M |
| ResNet50 | `resnet50` | 2048 | ~25M |

In [7]:
from medical_agent.core.ensemble_classifier import EfficientNetModel, MobileNetModel, ResNetModel

for ModelClass, name in [
    (EfficientNetModel, 'EfficientNet-B3'),
    (MobileNetModel, 'MobileNetV2'),
    (ResNetModel, 'ResNet50'),
]:
    model = ModelClass(num_classes=10, freeze_base=True)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{name}:')
    print(f'  Total parameters    : {total:,}')
    print(f'  Trainable (head)    : {trainable:,}')
    print()

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 192MB/s]


EfficientNet-B3:
  Total parameters    : 11,618,610
  Trainable (head)    : 922,378

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 122MB/s]


MobileNetV2:
  Total parameters    : 3,015,178
  Trainable (head)    : 791,306

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 220MB/s]


ResNet50:
  Total parameters    : 24,692,554
  Trainable (head)    : 1,184,522



## 5. Training Loop Utilities

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler, random_split
from torchvision import datasets
from medical_agent.core.image_pipeline import ImagePipeline
from medical_agent.core.skin_disease_classifier import FOLDER_TO_DISEASE

BATCH_SIZE = 32
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 30
LR = 1e-3
OUTPUT_DIR = '/content/drive/MyDrive/medical_vision_ai/models'
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pipeline = ImagePipeline()

def build_loaders(data_dir, batch_size=BATCH_SIZE):
    """Build train and validation DataLoaders."""
    train_tf = pipeline.get_train_transforms()
    val_tf = pipeline.get_val_transforms()

    full_ds = datasets.ImageFolder(root=data_dir, transform=train_tf)
    n_classes = len(full_ds.classes)
    class_counts = [0] * n_classes
    for _, lbl in full_ds.samples:
        class_counts[lbl] += 1

    n_train = int(0.8 * len(full_ds))
    n_val = len(full_ds) - n_train
    gen = torch.Generator().manual_seed(42)
    train_sub, val_sub = random_split(full_ds, [n_train, n_val], generator=gen)

    val_ds = datasets.ImageFolder(root=data_dir, transform=val_tf)
    val_final = Subset(val_ds, val_sub.indices)

    train_cc = [0] * n_classes
    for i in train_sub.indices:
        train_cc[full_ds.samples[i][1]] += 1
    sw = [1.0 / train_cc[full_ds.samples[i][1]] for i in train_sub.indices]
    sampler = WeightedRandomSampler(weights=sw, num_samples=len(sw), replacement=True)

    train_loader = DataLoader(train_sub, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_final, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    cw = torch.tensor([n_train / (n_classes * max(c, 1)) for c in train_cc], dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)

    return train_loader, val_loader, criterion, n_classes, full_ds.classes

def run_epoch(model, loader, criterion, optimizer, is_train):
    model.train() if is_train else model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if is_train and optimizer:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if is_train and optimizer:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            total_correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / max(total, 1), total_correct / max(total, 1)

print('✅ Training utilities ready.')

✅ Training utilities ready.


## 6. Train EfficientNet-B3

In [9]:
from medical_agent.core.ensemble_classifier import EfficientNetModel

train_loader, val_loader, criterion, n_classes, class_names = build_loaders(DATA_DIR)
efficientnet_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

effnet = EfficientNetModel(num_classes=n_classes, freeze_base=True).to(device)
best_val_acc_eff = 0.0
OUTPUT_EFFNET = os.path.join(OUTPUT_DIR, 'efficientnet_skin.pth')

# Phase 1: head only
print('Phase 1 — EfficientNet-B3 (head only)')
opt1 = torch.optim.Adam(filter(lambda p: p.requires_grad, effnet.parameters()), lr=LR)
for epoch in range(1, PHASE1_EPOCHS + 1):
    tl, ta = run_epoch(effnet, train_loader, criterion, opt1, True)
    vl, va = run_epoch(effnet, val_loader, criterion, None, False)
    efficientnet_history['train_loss'].append(round(tl, 4))
    efficientnet_history['val_loss'].append(round(vl, 4))
    efficientnet_history['train_acc'].append(round(ta, 4))
    efficientnet_history['val_acc'].append(round(va, 4))
    icon = '✅' if va > best_val_acc_eff else '  '
    print(f'  [P1] Epoch {epoch:2d}/{PHASE1_EPOCHS} | train {ta:.2%} | val {va:.2%} {icon}')
    if va > best_val_acc_eff:
        best_val_acc_eff = va
        torch.save(effnet.state_dict(), OUTPUT_EFFNET)

# Phase 2: fine-tune
print('\nPhase 2 — EfficientNet-B3 (full fine-tune)')
effnet.unfreeze_backbone()
backbone_p = [p for n, p in effnet.named_parameters() if 'classifier' not in n]
head_p     = [p for n, p in effnet.named_parameters() if 'classifier' in n]
opt2 = torch.optim.Adam([{'params': backbone_p, 'lr': LR*0.01}, {'params': head_p, 'lr': LR*0.1}])
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='max', patience=5, factor=0.5)
no_improve = 0
for epoch in range(1, PHASE2_EPOCHS + 1):
    tl, ta = run_epoch(effnet, train_loader, criterion, opt2, True)
    vl, va = run_epoch(effnet, val_loader, criterion, None, False)
    sched.step(va)
    efficientnet_history['train_loss'].append(round(tl, 4))
    efficientnet_history['val_loss'].append(round(vl, 4))
    efficientnet_history['train_acc'].append(round(ta, 4))
    efficientnet_history['val_acc'].append(round(va, 4))
    icon = '✅' if va > best_val_acc_eff else '  '
    print(f'  [P2] Epoch {epoch:2d}/{PHASE2_EPOCHS} | train {ta:.2%} | val {va:.2%} {icon}')
    if va > best_val_acc_eff:
        best_val_acc_eff = va; no_improve = 0
        torch.save(effnet.state_dict(), OUTPUT_EFFNET)
    else:
        no_improve += 1
        if no_improve >= 8:
            print(f'  ⏹️ Early stopping at epoch {epoch}')
            break

print(f'\n✅ EfficientNet-B3 best val acc: {best_val_acc_eff:.2%}')
print(f'   Saved to: {OUTPUT_EFFNET}')

FileNotFoundError: [Errno 2] No such file or directory: '/content/skin_data/IMG_CLASSES'

## 7. Train MobileNetV2

In [ ]:
from medical_agent.core.ensemble_classifier import MobileNetModel

mobilenet_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
mobilenet = MobileNetModel(num_classes=n_classes, freeze_base=True).to(device)
best_val_acc_mob = 0.0
OUTPUT_MOB = os.path.join(OUTPUT_DIR, 'mobilenet_skin.pth')

print('Phase 1 — MobileNetV2 (head only)')
opt1 = torch.optim.Adam(filter(lambda p: p.requires_grad, mobilenet.parameters()), lr=LR)
for epoch in range(1, PHASE1_EPOCHS + 1):
    tl, ta = run_epoch(mobilenet, train_loader, criterion, opt1, True)
    vl, va = run_epoch(mobilenet, val_loader, criterion, None, False)
    mobilenet_history['train_loss'].append(round(tl, 4))
    mobilenet_history['val_loss'].append(round(vl, 4))
    mobilenet_history['train_acc'].append(round(ta, 4))
    mobilenet_history['val_acc'].append(round(va, 4))
    icon = '✅' if va > best_val_acc_mob else '  '
    print(f'  [P1] Epoch {epoch:2d}/{PHASE1_EPOCHS} | train {ta:.2%} | val {va:.2%} {icon}')
    if va > best_val_acc_mob:
        best_val_acc_mob = va
        torch.save(mobilenet.state_dict(), OUTPUT_MOB)

print('\nPhase 2 — MobileNetV2 (full fine-tune)')
mobilenet.unfreeze_backbone()
backbone_p = [p for n, p in mobilenet.named_parameters() if 'classifier' not in n]
head_p     = [p for n, p in mobilenet.named_parameters() if 'classifier' in n]
opt2 = torch.optim.Adam([{'params': backbone_p, 'lr': LR*0.01}, {'params': head_p, 'lr': LR*0.1}])
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='max', patience=5, factor=0.5)
no_improve = 0
for epoch in range(1, PHASE2_EPOCHS + 1):
    tl, ta = run_epoch(mobilenet, train_loader, criterion, opt2, True)
    vl, va = run_epoch(mobilenet, val_loader, criterion, None, False)
    sched.step(va)
    mobilenet_history['train_loss'].append(round(tl, 4))
    mobilenet_history['val_loss'].append(round(vl, 4))
    mobilenet_history['train_acc'].append(round(ta, 4))
    mobilenet_history['val_acc'].append(round(va, 4))
    icon = '✅' if va > best_val_acc_mob else '  '
    print(f'  [P2] Epoch {epoch:2d}/{PHASE2_EPOCHS} | train {ta:.2%} | val {va:.2%} {icon}')
    if va > best_val_acc_mob:
        best_val_acc_mob = va; no_improve = 0
        torch.save(mobilenet.state_dict(), OUTPUT_MOB)
    else:
        no_improve += 1
        if no_improve >= 8:
            print(f'  ⏹️ Early stopping at epoch {epoch}')
            break

print(f'\n✅ MobileNetV2 best val acc: {best_val_acc_mob:.2%}')
print(f'   Saved to: {OUTPUT_MOB}')

## 8. Train ResNet50

In [ ]:
from medical_agent.core.ensemble_classifier import ResNetModel

resnet_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
resnet = ResNetModel(num_classes=n_classes, freeze_base=True).to(device)
best_val_acc_res = 0.0
OUTPUT_RES = os.path.join(OUTPUT_DIR, 'resnet_skin.pth')

print('Phase 1 — ResNet50 (head only)')
opt1 = torch.optim.Adam(filter(lambda p: p.requires_grad, resnet.parameters()), lr=LR)
for epoch in range(1, PHASE1_EPOCHS + 1):
    tl, ta = run_epoch(resnet, train_loader, criterion, opt1, True)
    vl, va = run_epoch(resnet, val_loader, criterion, None, False)
    resnet_history['train_loss'].append(round(tl, 4))
    resnet_history['val_loss'].append(round(vl, 4))
    resnet_history['train_acc'].append(round(ta, 4))
    resnet_history['val_acc'].append(round(va, 4))
    icon = '✅' if va > best_val_acc_res else '  '
    print(f'  [P1] Epoch {epoch:2d}/{PHASE1_EPOCHS} | train {ta:.2%} | val {va:.2%} {icon}')
    if va > best_val_acc_res:
        best_val_acc_res = va
        torch.save(resnet.state_dict(), OUTPUT_RES)

print('\nPhase 2 — ResNet50 (full fine-tune)')
resnet.unfreeze_backbone()
backbone_p = [p for n, p in resnet.named_parameters() if not n.startswith('base.fc')]
head_p     = [p for n, p in resnet.named_parameters() if n.startswith('base.fc')]
opt2 = torch.optim.Adam([{'params': backbone_p, 'lr': LR*0.01}, {'params': head_p, 'lr': LR*0.1}])
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='max', patience=5, factor=0.5)
no_improve = 0
for epoch in range(1, PHASE2_EPOCHS + 1):
    tl, ta = run_epoch(resnet, train_loader, criterion, opt2, True)
    vl, va = run_epoch(resnet, val_loader, criterion, None, False)
    sched.step(va)
    resnet_history['train_loss'].append(round(tl, 4))
    resnet_history['val_loss'].append(round(vl, 4))
    resnet_history['train_acc'].append(round(ta, 4))
    resnet_history['val_acc'].append(round(va, 4))
    icon = '✅' if va > best_val_acc_res else '  '
    print(f'  [P2] Epoch {epoch:2d}/{PHASE2_EPOCHS} | train {ta:.2%} | val {va:.2%} {icon}')
    if va > best_val_acc_res:
        best_val_acc_res = va; no_improve = 0
        torch.save(resnet.state_dict(), OUTPUT_RES)
    else:
        no_improve += 1
        if no_improve >= 8:
            print(f'  ⏹️ Early stopping at epoch {epoch}')
            break

print(f'\n✅ ResNet50 best val acc: {best_val_acc_res:.2%}')
print(f'   Saved to: {OUTPUT_RES}')

## 9. Training Visualisation — Loss & Accuracy Graphs

In [ ]:
import json

all_histories = {
    'EfficientNet-B3': efficientnet_history,
    'MobileNetV2':     mobilenet_history,
    'ResNet50':        resnet_history,
}

history_path = os.path.join(OUTPUT_DIR, 'ensemble_training_history.json')
with open(history_path, 'w') as f:
    json.dump(all_histories, f, indent=2)
print(f'✅ Training history saved to {history_path}')

In [ ]:
import sys
sys.path.insert(0, '/content/medical_agent_starter')
from scripts.plot_training_history import plot_all

OUTPUT_PLOTS = '/content/drive/MyDrive/medical_vision_ai/plots'
os.makedirs(OUTPUT_PLOTS, exist_ok=True)

plot_all(history_path, output_dir=OUTPUT_PLOTS)
print(f'\n✅ Plots saved to {OUTPUT_PLOTS}')

In [ ]:
# Show inline comparison plot
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt

colors = {'EfficientNet-B3': '#2196F3', 'MobileNetV2': '#4CAF50', 'ResNet50': '#FF5722'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Ensemble Models — Validation Comparison', fontsize=14, fontweight='bold')

for name, hist in all_histories.items():
    epochs = range(1, len(hist['val_loss']) + 1)
    color = colors[name]
    axes[0].plot(epochs, hist['val_loss'], label=name, color=color, linewidth=2)
    axes[1].plot(epochs, [a*100 for a in hist['val_acc']], label=name, color=color, linewidth=2)

for ax, title, ylabel in zip(axes, ['Validation Loss', 'Validation Accuracy (%)'], ['Loss', 'Accuracy (%)']):
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Ensemble Inference Demo

In [ ]:
from medical_agent.core.ensemble_classifier import EnsembleClassifier

ensemble = EnsembleClassifier(
    efficientnet_path=OUTPUT_EFFNET,
    mobilenet_path=OUTPUT_MOB,
    resnet_path=OUTPUT_RES,
)

print(f'Available models: {ensemble.available_models}')

# Load a sample image for inference
if os.path.exists(DATA_DIR):
    import random
    from PIL import Image
    folders = [f for f in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, f))]
    folder = random.choice(folders)
    imgs = [f for f in os.listdir(os.path.join(DATA_DIR, folder))
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    img_path = os.path.join(DATA_DIR, folder, random.choice(imgs))
    test_image = Image.open(img_path).convert('RGB')

    print(f'\nTest image: {img_path}')

    # Run ensemble inference
    result = ensemble.predict(test_image, top_n=5)
    individual = ensemble.get_individual_predictions(test_image, top_n=3)

    print(f'\n🏆 Ensemble Result:')
    print(f'   Top diagnosis : {result.top_diagnosis.readable_name}')
    print(f'   Confidence    : {result.top_diagnosis.confidence:.1%}')
    print(f'   Severity      : {result.top_diagnosis.severity}')
    print(f'   Urgent        : {result.needs_urgent_attention}')

    print('\n🔍 Individual Model Predictions:')
    for model_name, preds in individual.items():
        print(f'  {model_name}: {preds[0].readable_name} ({preds[0].confidence:.1%})')

    # Show image and results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(test_image)
    ax1.set_title('Test Image')
    ax1.axis('off')

    names = [c.readable_name[:20] for c in result.candidates]
    confs = [c.confidence * 100 for c in result.candidates]
    colors_bar = ['red' if c.urgency == 'critique' else
                  'orange' if c.urgency == 'urgent' else
                  'gold' if c.urgency == 'modéré' else 'green'
                  for c in result.candidates]
    ax2.barh(names[::-1], confs[::-1], color=colors_bar[::-1])
    ax2.set_xlabel('Confidence (%)')
    ax2.set_title('Ensemble Predictions')
    ax2.set_xlim(0, 100)
    plt.tight_layout()
    plt.show()

## 11. VLM Explanation Demo

In [ ]:
from medical_agent.core.vlm_explainer import VLMExplainer

# Use template-based fallback (no internet required)
explainer = VLMExplainer(use_blip=False)

if os.path.exists(DATA_DIR):
    explanation = explainer.explain(test_image, result)
    print('💬 VLM Explanation:')
    print('=' * 60)
    print(explanation)
    print('=' * 60)
else:
    print('⚠️ No test image available. Run the ensemble inference cell first.')

## 12. Save Models to Google Drive

In [ ]:
import shutil

# Models are already saved to OUTPUT_DIR which is in Google Drive
print('Models in Google Drive:')
for fname in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname:40s} {size_mb:.1f} MB')

print(f'\n✅ All models saved to: {OUTPUT_DIR}')
print('\nTo use in your local project, download and place in the models/ directory.')

## 13. Conclusion

### What we built
- ✅ **Three trained deep learning models** for skin disease classification:
  - EfficientNet-B3, MobileNetV2, ResNet50 — all with ImageNet pre-training and custom heads
- ✅ **Ensemble classifier** with consensus / disagreement logic
- ✅ **VLM explanation module** for natural language diagnosis summaries
- ✅ **Training history** with loss and accuracy plots for all 3 models

### How to use in production
1. Copy trained `.pth` files to the `models/` directory in the project
2. Run the Streamlit app: `streamlit run app.py`
3. Upload a skin image in the **📷 Diagnostic Peau** tab
4. The ensemble will predict, individual model results are shown, and VLM explanation is generated

### Next steps
- Train with more data or more epochs for higher accuracy
- Add Grad-CAM visualisations to highlight the regions the model focuses on
- Integrate a real VLM (e.g., LLaVA, BLIP-2) for richer explanations
- Add test-time augmentation (TTA) to the ensemble for more robust predictions

---
> ⚠️ **Medical Disclaimer**: This system is for research and educational purposes only.
> It does **NOT** provide medical diagnoses. Always consult a licensed healthcare professional.
> ⚠️ **This is not medical advice. Consult a healthcare professional.**